# Part B: Agentic Data Mining with ReAct

In [3]:
!python --version

# min requirements
# • Python ≥ 3.10
# • transformers, accelerate, torch
# • pandas, numpy
# • pydantic
# • outlines

# # Safety notes
# • Do not allow internet access from the execution environment
# • Restrict code to reading the provided CSV files only
# • Avoid writing files, deleting files, or calling system commands

Python 3.12.12


## Task 1 — Dataset Inspection and Sanity Checks

### Task 1.1 Load and Inspect JSONL Files

In [4]:
# QUESTION 14: Load da-dev-questions.jsonl and da-dev-labels.jsonl. Report:
# • The number of questions and labels
# • The set of keys present in a question record (print one example)
# • The set of keys present in a label record (print one example)

from google.colab import drive
drive.mount('/content/drive')

file_path = '/content/drive/My Drive/EC ENGR 219 Large-Scale Data Mining: Models & Algorithms/Project 3/ECE_219_Wintern26_Project3_data_helpercode/ECE_219_Wintern26_Project3_data_helpercode/proj3_partb_partc_agent_data/share_data'
file_path_questions = file_path + '/da-dev-questions.jsonl'
file_path_labels = file_path + '/da-dev-labels.jsonl'
file_path_tables = file_path + '/da-dev-tables/'

import json
import pandas as pd

# Load questions, one record at at a time using .jsonl
questions = []
with open(file_path_questions, 'r') as f:
    for line in f:
        questions.append(json.loads(line))

# Load labels
labels = []
with open(file_path_labels, 'r') as f:
    for line in f:
        labels.append(json.loads(line))

# Number of questions & labels
print(f"Number of questions: {len(questions)}")
print(f"Number of labels: {len(labels)}")

# See keys in a question record
print(f"\nKeys in a question record: {list(questions[0].keys())}")
print(f"\nExample question record:")
print(json.dumps(questions[0], indent=2)) #indent key-value pairs by 2 indents; "concepts" is used as metadata

# See keys in a label record
print(f"\nKeys in a label record: {list(labels[0].keys())}")
print(f"\nExample label record:")
print(json.dumps(labels[0], indent=2)) #indent key-value pairs by 2 indents; contains both "key" and "value"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Number of questions: 257
Number of labels: 257

Keys in a question record: ['id', 'question', 'concepts', 'constraints', 'format', 'file_name', 'level']

Example question record:
{
  "id": 0,
  "question": "Calculate the mean fare paid by the passengers.",
  "concepts": [
    "Summary Statistics"
  ],
  "constraints": "Calculate the mean fare using Python's built-in statistics module or appropriate statistical method in pandas. Rounding off the answer to two decimal places.",
  "format": "@mean_fare[mean_fare_value] where \"mean_fare_value\" is a floating-point number rounded to two decimal places.",
  "file_name": "test_ave.csv",
  "level": "easy"
}

Keys in a label record: ['id', 'common_answers']

Example label record:
{
  "id": 0,
  "common_answers": [
    [
      "mean_fare",
      "34.65"
    ]
  ]
}


Task 1.2 Inspect the Table References

In [5]:
# QUESTION 15: Pick 3 random question IDs. For each:
# • Print the file name of the referenced CSV
# • Load the CSV with pandas and print df.shape, df.dtypes, and df.head(3)
# • Print the corresponding question

import random
random.seed(23)
random_ids = random.sample(range(len(questions)), 3)

for idx in random_ids:
    q = questions[idx]

    print(f"Question ID: {q['id']}")
    print(f"Question: {q['question']}")
    print(f"File name: {q['file_name']}")

    # Load CSV
    csv_path = file_path_tables + q['file_name']
    df = pd.read_csv(csv_path)

    print(f"\n df.shape: {df.shape}")
    print(f"\n df.dtypes:\n{df.dtypes}")
    print(f"\n df.head(3):\n{df.head(3)}")
    print("\n")

Question ID: 452
Question: 1. Is there a relationship between wind speed (WINDSPEED) and atmospheric pressure (BARO) for wind direction (DIR) equal to 180 degrees? Calculate the Pearson correlation coefficient for this specific wind direction.
File name: baro_2015.csv

 df.shape: (8736, 8)

 df.dtypes:
DATE TIME      object
 WINDSPEED    float64
 DIR            int64
 GUSTS        float64
 AT           float64
 BARO         float64
 RELHUM       float64
 VIS          float64
dtype: object

 df.head(3):
          DATE TIME   WINDSPEED   DIR   GUSTS    AT    BARO   RELHUM   VIS
0  01/01/2015 00:00        2.72   288    5.25  27.7  1023.0      NaN   NaN
1  01/01/2015 01:00        3.89   273    7.00  26.8  1022.7      NaN   NaN
2  01/01/2015 02:00        4.86   268    6.41  27.0  1022.1      NaN   NaN


Question ID: 114
Question: Which country has the highest happiness score?
File name: 2015.csv

 df.shape: (158, 12)

 df.dtypes:
Country                           object
Region              

Task 1.3 Understand the Required Answer Format

In [6]:
# QUESTION 16: Find 2 examples where the required format contains multiple answer slots
 # (e.g., two or more @name[value] fields). Explain:
# • How the dataset represents multi-part answers in the labels
# • How you plan to evaluate such answers automatically

# Manually identified question IDs 6 & 8 as having multi-part answers
for qid in [6, 8]:
    q = next(q for q in questions if q['id'] == qid)
    label = next(l for l in labels if l['id'] == qid)

    print(f"Question ID:  {q['id']}")
    print(f"Question:     {q['question']}")
    print(f"Format:       {q['format']}")
    print(f"Label:        {json.dumps(label, indent=2)}")
    print(f"\n")

# Data represents multi-part answers in each label record, through common_answers field with a list of [key, value] pairs - for example:
# "common_answers": [
#   [
#     "mean_fare_elderly",
#     "43.47"
#   ],
#   [
#     "mean_fare_teenager",
#     "31.98"
#   ]
# ]



# To evaluate the multi-part answers automatically, check each [key, value] pair in common_answers independently against the agent's output
# After matching question with label via 'id', then iterate through common_answers field and check if each @key[value] appears in agent output

# Example: agent is given format "@mean_fare_elderly[value] @mean_fare_teenager[value]"
# Agent output may look like: @mean_fare_elderly[45.32] @mean_fare_teenager[12.87] @mean_fare_child[8.50]
# Agent output is then checked and compared to see if it appears in list of common_answers field

Question ID:  6
Question:     Create a new column called "AgeGroup" that categorizes the passengers into four age groups: 'Child' (0-12 years old), 'Teenager' (13-19 years old), 'Adult' (20-59 years old), and 'Elderly' (60 years old and above). Then, calculate the mean fare for each age group.
Format:       @mean_fare_child[mean_fare], @mean_fare_teenager[mean_fare], @mean_fare_adult[mean_fare], @mean_fare_elderly[mean_fare], where "mean_fare" is a float number rounded to 2 decimal places.
Label:        {
  "id": 6,
  "common_answers": [
    [
      "mean_fare_elderly",
      "43.47"
    ],
    [
      "mean_fare_teenager",
      "31.98"
    ],
    [
      "mean_fare_child",
      "31.09"
    ],
    [
      "mean_fare_adult",
      "35.17"
    ]
  ]
}


Question ID:  8
Question:     Perform a distribution analysis on the 'Fare' column for each passenger class ('Pclass') separately. Calculate the mean, median, and standard deviation of the fare for each class. Interpret the results in t

Task 1.4 Checking the subset

In [7]:
# QUESTION 17: Unfortunately, the model we are going to use is still not powerful enough to solve all the tasks
# Here we are selecting 10 sub-tasks that are proved to be solveable:
# The selected IDs are: SELECTED IDS = [0, 5, 9, 10, 14, 18, 24, 25, 26, 55]
# • Print out and check those tasks

SELECTED_IDS = [0, 5, 9, 10, 14, 18, 24, 25, 26, 55]

selected_questions = [q for q in questions if q['id'] in SELECTED_IDS]
selected_labels = [l for l in labels if l['id'] in SELECTED_IDS]


for q in selected_questions:
    label = next(l for l in selected_labels if l['id'] == q['id']) # next returns first item, from generator

    print(f"ID:          {q['id']}")
    print(f"Level:       {q['level']}")
    print(f"Concepts:    {q['concepts']}")
    print(f"Question:    {q['question']}")
    print(f"Constraints: {q['constraints']}")
    print(f"Format:      {q['format']}")
    print(f"File:        {q['file_name']}")
    print(f"Answer:      {label['common_answers']}")
    print(f"\n")

  # Functions: mean/median/stddev, correlation_coefficient, check normal distr.

ID:          0
Level:       easy
Concepts:    ['Summary Statistics']
Question:    Calculate the mean fare paid by the passengers.
Constraints: Calculate the mean fare using Python's built-in statistics module or appropriate statistical method in pandas. Rounding off the answer to two decimal places.
Format:      @mean_fare[mean_fare_value] where "mean_fare_value" is a floating-point number rounded to two decimal places.
File:        test_ave.csv
Answer:      [['mean_fare', '34.65']]


ID:          5
Level:       medium
Concepts:    ['Feature Engineering', 'Correlation Analysis']
Question:    Generate a new feature called "FamilySize" by summing the "SibSp" and "Parch" columns. Then, calculate the Pearson correlation coefficient (r) between the "FamilySize" and "Fare" columns.
Constraints: Create a new column 'FamilySize' that is the sum of 'SibSp' and 'Parch' for each row.
Calculate the Pearson correlation coefficient between 'FamilySize' and 'Fare'
Do not perform any further data clea

## Task 2 — Model Loading and Structured Output: Make the Planner Parseable

In [8]:
!pip install -q transformers accelerate pydantic torch outlines

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.3 MB/s eta 0:00:00


In [9]:
import torch
import outlines
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507" # Small open-source language model made by Alibaba;
# base models predict next token, instruct models take base model and train specifically on prompt/response involving SFT/RLHF etc

hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
hf_model = AutoModelForCausalLM.from_pretrained(             # Loads actual model weights
    MODEL_NAME, device_map="auto", torch_dtype=torch.float16 # Load weight in half precision to save GPU memory
)
model = outlines.from_transformers(hf_model, hf_tokenizer)   # Instead of the model freely generating any tokens, Outlines forces token generation tha's valid according to Pydantic schema
tokenizer = hf_tokenizer
print(f"Model loaded on {hf_model.device}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Model loaded on cpu


In [10]:
# Helper code
from typing import Type, Optional
from pydantic import BaseModel
import re
from dataclasses import dataclass as dc

@dc #shorthand for data class
class LLMResponse:
    content: str
    raw: str

def generate(
    messages: list[dict],                               # conversation history e.g. [{"role": "user", "content": "..."}]
    response_format: Optional[Type[BaseModel]] = None,  # if provided then structured output, else if None then plain text
    max_new_tokens: int = 4096,                         # max tokens to generate
    temperature: float = 0.6,                           # randomness (0=deterministic, 1=creative)
    top_p: float = 0.95,                                # nucleus sampling threshold
) -> "LLMResponse | BaseModel":                         # quotations allow for lazy evaluation (e.g. if LLM response is defined after this function)
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # converts [{"role": "user", "content": "hello"}] into readable format e.g. "User: hello\nAssistant:"

    # Structured output - outlines guarantees valid JSON
    if response_format is not None:
        result = model(prompt, response_format, max_new_tokens=max_new_tokens)
        return response_format.model_validate_json(result)

    # Plain text — use transformers directly
    inputs = tokenizer(prompt, return_tensors="pt").to(hf_model.device)
    output_ids = hf_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
    )
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(generated_ids, skip_special_tokens=False)
    content = re.sub(r"<\|[^>]+\|>", "", raw).strip()
    return LLMResponse(content=content, raw=raw)

In [11]:
# QUESTION 18: Demonstrate (with 5 different prompts) that your planner always returns valid JSON that parses into your Pydantic model without try/except fallbacks
# Include at least one case where the planner decides it is done (is done=true)

# Define Pydantic schema for planner output
class PlannerOutput(BaseModel):
    thought: str      # planner's reasoning / thought process
    is_done: bool     # True if planner has final answer, else False if more steps needed
    response: str     # next instruction response for program if is_done=False, OR final response if is_done=True

# System prompt for planner
PLANNER_SYSTEM_PROMPT = """You are an experienced data analysis technical expert. Given a question about a CSV file,
decide the next analysis step or provide a final response to the question.
You must respond in valid JSON with fields: thought (str), is_done (bool), response (str)."""

In [ ]:
# 5 different prompts to demonstrate structured output validation
test_cases = [

    [
        {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
        {"role": "user", "content": "Question: What is the mean fare paid by passengers?\nCSV columns: PassengerId, Fare, Age, Survived"}
    ],

    [
        {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
        {"role": "user", "content": "Question: What is the average salary of employees in the Engineering department?\nCSV columns: EmployeeId, Department, Salary, Age"}
    ],

    [
        {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
        {"role": "user", "content": "Question: What is the percentage of passengers who survived?\nCSV columns: PassengerId, Survived, Age, Fare"}
    ],

    [
        {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
        {"role": "user", "content": "Question: What is the mean fare paid by passengers?\nPrevious step result: computed mean fare = 34.65\nFormat: @mean_fare[value]\nThe answer has already been computed, provide the final formatted answer."}
    ],

    [
        {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
        {"role": "user", "content": "Question: Which product had the highest total sales?\nCSV columns: ProductId, ProductName, Sales, Region"}
    ],
]

print("Demonstrating structured planner output across 5 tester prompts:")

for i, messages in enumerate(test_cases, 1):

    print(f"Prompt {i}:")
    print(f"Prompt {i}: {messages[1]['content'].split(chr(10))[0]}") # Take the question in "content" from user role, prior to specified newline char

    # use generate() from helper code with PlannerOutput schema
    # returns a validated PlannerOutput instance directly
    output = generate(messages, response_format=PlannerOutput)

    print(f"thought:   {output.thought}")
    print(f"is_done:   {output.is_done}")
    print(f"response:  {output.response}")

    # Validate output schema according to PlannerOutput
    assert isinstance(output, PlannerOutput), "Output is not a valid PlannerOutput"
    assert isinstance(output.thought, str), "thought must be a string"
    assert isinstance(output.is_done, bool), "is_done must be a bool"
    assert isinstance(output.response, str), "response must be a string"

    print(f"Valid PlannerOutput confirmed -> is_done={output.is_done}")

Demonstrating structured planner output across 5 tester prompts:
Prompt 1:
Prompt 1: Question: What is the mean fare paid by passengers?
thought:   The question asks for the mean fare paid by passengers. The relevant column in the CSV is 'Fare'. To answer this, I need to calculate the mean (average) value of the 'Fare' column.
is_done:   False
response:  I will calculate the mean of the 'Fare' column from the CSV data to determine the average fare paid by passengers.
Valid PlannerOutput confirmed -> is_done=False
Prompt 2:
Prompt 2: Question: What is the average salary of employees in the Engineering department?
thought:   The question asks for the average salary of employees in the Engineering department. To answer this, I need to filter the rows where the Department column has the value 'Engineering' and then compute the average of the Salary column for those rows.
is_done:   False
response:  I will filter the data to include only employees in the Engineering department and calculate

In [12]:
# QUESTION 19: Explain in a few sentences why structured output is useful for large-scale data mining pipelines

# Structured output is important for large-scale data mining pipelines because it guarantees that the LLM always returns responses that can be parsed by machines with correct data types
# This allows downstream components to ingest these responses without breakage, and without try & except fallbacks

# Two libraries work together to enforce this:
# - Outlines: blocks invalid tokens during generation, guaranteeing syntactically valid JSON
# - Pydantic: validates the JSON structure matches the defined schema (correct fields and data types)

# At large scale with thousands of CSV files and heterogeneous schemas, even a small parsing failure rate causes many silent failures that can be hard to debug
# Structured output eliminates this failure mode entirely by making invalid output structurally impossible at the token generation level

## Task 3 — Build a ReAct Data Analysis Agent

In [13]:
# Helper code
### --- Agent 1: Executor (Python sandbox) ---

import io
import sys
import traceback


class Executor:
    """Executes Python code with a persistent namespace across calls.

    The namespace is shared between runs, so variables defined in step 1
    are available in step 2, etc.  Common libraries are pre-imported.
    """

    def __init__(self):
        self.namespace = {"__builtins__": __builtins__}
        # Pre-import libraries the coder will likely use
        exec(
            "import pandas as pd\nimport numpy as np\n"
            "from scipy import stats\nfrom sklearn import *\n",
            self.namespace,
        )

    def run(self, code: str, timeout: int = 30) -> str:
        """Execute code and return captured stdout/stderr."""
        stdout_buf = io.StringIO()
        stderr_buf = io.StringIO()
        old_stdout, old_stderr = sys.stdout, sys.stderr
        sys.stdout, sys.stderr = stdout_buf, stderr_buf

        try:
            exec(code, self.namespace)
        except Exception:
            traceback.print_exc(file=stderr_buf)
        finally:
            sys.stdout, sys.stderr = old_stdout, old_stderr

        stdout = stdout_buf.getvalue().strip()
        stderr = stderr_buf.getvalue().strip()

        if stderr:
            return f"[STDOUT]\n{stdout}\n\n[ERROR]\n{stderr}" if stdout else f"[ERROR]\n{stderr}"
        return stdout if stdout else "(no output)"

    def reset(self):
        """Clear state for a new task."""
        self.__init__()


# Quick test
exe = Executor()
print(exe.run("x = 40 + 2\nprint(f'The answer is {x}')"))
print(exe.run("print(x * 2)"))  # x persists from previous run

The answer is 42
84


In [16]:
from pydantic import BaseModel
from typing import Optional
import pandas as pd

# Define Pydantic schema for planner output
class PlannerOutput(BaseModel):
    thought: str      # planner's reasoning / thought process
    is_done: bool     # True if planner has final answer, else False if more steps needed
    response: str     # next instruction response for program if is_done=False, OR final response if is_done=True

class ObserverOutput(BaseModel):
    summary: str         # concise summary of what has happened
    is_error: bool       # True, if code produced an error
    next_hint: str       # suggestion for planner's next step

# Define system prompts

PLANNER_SYSTEM_PROMPT = """You are a technical data analyst expert. You are given a question about a CSV file.
Decide the next analysis step, or if you have enough information, provide the final formatted answer.
The CSV file is already loaded as a pandas DataFrame called `df`.
You must respond in valid JSON with fields: thought (str), is_done (bool), response (str).
If is_done=true, response must contain the final answer in the exact format specified.
If is_done=false, response must contain a clear instruction for the coder."""

CODER_SYSTEM_PROMPT = """You are a technical data analyst expert in Python. Write clean, concise pandas code to complete the instruction.
The CSV is already loaded as `df`. Do not reload it.
Only output raw Python code with no explanation, markdown or backticks.
Always print your code result so it appears in stdout."""

# Stdout records the stream of print output chronologically

OBSERVER_SYSTEM_PROMPT = """You are a technical data analyst expert in observation and review mode. Given the code output and original instruction,
summarize what happened in maximum 3 sentences. Note any errors and suggest a coding fix, if required.
Always respond with JSON containing: summary (str), is_error (bool), next_hint (str)."""


# Define agent components

def run_planner(question, constraints, fmt, history, csv_summary):
    """Planner to decide next iter or returns final, formatted answer"""

    # bounded to last 5 iterations to manage context window
    history_str = ""
    for i, iter in enumerate(history[-5:], 1):
        history_str += f"\nIteration {i}:"
        history_str += f"\nPlan: {iter['iter']}"
        history_str += f"\nObservation: {iter['observation']}"

    messages = [
        {"role": "system", "content": PLANNER_SYSTEM_PROMPT},
        {"role": "user", "content":
            f"Question: {question}\n"
            f"Constraints: {constraints}\n"
            f"Required format: {fmt}\n"
            f"CSV summary:\n{csv_summary}\n"
            f"History:{history_str if history_str else 'None yet'}\n"
            f"What is the next iteration step?"
        }
    ]
    return generate(messages, response_format=PlannerOutput, max_new_tokens=512)


def run_coder(plan, csv_summary):
    """Coder writes Python code based on planner's instruction"""
    messages = [
        {"role": "system", "content": CODER_SYSTEM_PROMPT},
        {"role": "user", "content":
            f"Plan: {plan}\n"
            f"CSV summary (for schema reference):\n{csv_summary}\n"
            f"Write the Python code:"
        }
    ]
    resp = generate(messages, max_new_tokens=512)

    # clean any: spaces, newlines, tabs, markdown, backticks
    code = resp.content.strip()
    code = code.replace("```python", "").replace("```", "").strip()

    return code


def run_observer(plan, code, raw_output):
    """Observer summarises raw executor output for observation/review"""
    messages = [
        {"role": "system", "content": OBSERVER_SYSTEM_PROMPT},
        {"role": "user", "content":
            f"Plan: {plan}\n"
            f"Code:\n{code}\n"
            f"Raw output (truncated to 500 chars):\n{raw_output[:500]}\n"
            f"Summarise what happened:"
        }
    ]
    return generate(messages, response_format=ObserverOutput, max_new_tokens=256)


def get_csv_summary(csv_path):
    """Load CSV and summarise: shape, data types & row preview"""
    df = pd.read_csv(csv_path)
    preview = f"Shape: {df.shape}\n"
    preview += f"Summary stats:\n{df.describe().to_string()}\n"
    preview += f"Columns + dtypes:\n{df.dtypes.to_string()}\n"
    preview += f"First 5 rows:\n{df.head(5).to_string()}"
    return preview


# Main ReAct Agent call

def run_react_agent(question_record, label_record, tables_path, max_iter=5):
    """Run complete ReAct loop for one question."""

    question    = question_record['question']
    constraints = question_record['constraints']
    fmt         = question_record['format']
    file_name   = question_record['file_name']
    qid         = question_record['id']
    csv_path    = tables_path + file_name

    print(f"Q{qid}: {question}")
    print(f"Format: {fmt}")
    print(f"File: {file_name}")

    # get CSV summary
    csv_summary = get_csv_summary(csv_path)

    # use helper Executor that pre-imports pandas, numpy, scipy, sklearn
    # load CSV df into executor persistent namespace, along with built ins and pre-imported libraries
    executor = Executor()
    executor.run(f"df = pd.read_csv('{csv_path}')")

    history = []
    final_answer = None

    for iter in range(max_iter):
        print(f"\nIteration {iter+1}")

        # Step 1: Planner
        planner_output = run_planner(question, constraints, fmt, history, csv_summary)
        print(f"[Planner] thought:   {planner_output.thought}")
        print(f"[Planner] is_done:   {planner_output.is_done}")
        print(f"[Planner] response:  {planner_output.response}")

        # # Step2: Check if Done
        if planner_output.is_done:
            final_answer = planner_output.response
            print(f"\nFinal answer: {final_answer}")
            break

        # Step 3: Python coder
        plan = planner_output.response
        code = run_coder(plan, csv_summary)
        print(f"[Coder] code:\n{code}")

        # Step 4: Executor- runs code in persistent namespace, df already loaded
        raw_output = executor.run(code)
        print(f"[Executor] output: {raw_output[:500]}")

        # Step 5: Code review, upon discovery of error
        if "[ERROR]" in raw_output:
            print(f"[Executor] error detected, retrying...")
            retry_instruction = (
                f"The previous code produced an error:\n{raw_output[:500]}\n"
                f"Original plan: {plan}\n"
                f"Please fix the code, and try again."
            )
            code = run_coder(retry_instruction, csv_summary)
            print(f"[Coder] retry code:\n{code}")
            raw_output = executor.run(code)
            print(f"[Executor] retry output: {raw_output[:500]}")

        # Step 6: Observer - summarises raw output into concise observation/review
        observer_output = run_observer(plan, code, raw_output)
        print(f"[Observer] summary:        {observer_output.summary}")
        print(f"[Observer] is_error:       {observer_output.is_error}")
        print(f"[Observer] next_hint:      {observer_output.next_hint}")

        # Step 7: Append to history - and truncate to bound context window
        history.append({
            "plan": plan,
            "code": code[:300],            # truncate code in history
            "raw_output": raw_output[:300], # truncate output in history
            "observation": observer_output.summary
        })

        # reset executor between steps to avoid namespace pollution
        # but re-load df since it's needed across all steps
        # NOTE: we do NOT reset - persistent namespace is a feature not a bug
        # df and intermediate variables persist across steps intentionally

    if final_answer is None:
        final_answer = "Agent did not converge within max iterations - unsuccessful"
        print(final_answer)

    return final_answer


In [ ]:
# Evaluate on 10 selected tasks

def evaluate_answer(agent_output, label):
    """Check each @key[value] pair from ground truth against agent output"""
    correct = 0
    total = len(label['common_answers'])
    for name, value in label['common_answers']:
        expected = f"@{name}[{value}]"
        if expected in agent_output:
            correct += 1
    return correct / total


# Run on all 10 selected tasks
tables_path = file_path_tables
results = []

for question in selected_questions:
    label = next(l for l in selected_labels if l['id'] == q['id'])

    # reset executor state between questions
    agent_answer = run_react_agent(question, label, tables_path, max_iter=5)
    score = evaluate_answer(agent_answer, label)

    results.append({
        "id":           q['id'],
        "question":     q['question'],
        "agent_answer": agent_answer,
        "ground_truth": label['common_answers'],
        "score":        score
    })

    print(f"\nScore for Q{q['id']}: {score:.2f}")

# Final accuracy
accuracy = sum(r['score'] for r in results) / len(results)
print(f"Final Accuracy: {accuracy:.2%} ({sum(r['score'] for r in results):.1f}/{len(results)})")

# Breakdown of reults, per question
print("\nPer question results:")
for r in results:
    status = "PASSED" if r['score'] == 1.0 else "PARTIAL PASS" if r['score'] > 0 else "FAILED"
    print(f"{status} Q{r['id']}: score={r['score']:.2f} | agent_ans={r['agent_answer'][:100]} | ground_truth={r['ground_truth']}")

Q0: Calculate the mean fare paid by the passengers.
Format: @mean_fare[mean_fare_value] where "mean_fare_value" is a floating-point number rounded to two decimal places.
File: test_ave.csv

Iteration 1
